In [ ]:
import pandas as pd


In [ ]:
# Cargar datos desde el archivo CSV
from pathlib import Path
import pandas as pd

csv_path = Path("data/query_actividades_proc_q_actual.csv")
df = pd.read_csv(csv_path)

print(f"Archivo cargado desde: {csv_path}")
print(f"DataFrame cargado con {len(df)} filas y {len(df.columns)} columnas")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras filas:")
df.head()
df.info()


In [16]:
# Cargar arquetipos y hacer matching con los resultados SQL
import sys, os
# Asegura que la carpeta del notebook esté en sys.path
sys.path.insert(0, os.getcwd())

# Intentar import normal, con fallback a carga por ruta si falla
try:
    from helper.match_arquetipos import load_arquetipos, match_codes
except ModuleNotFoundError:
    import importlib.util
    mod_path = os.path.join(os.getcwd(), 'helper', 'match_arquetipos.py')
    spec = importlib.util.spec_from_file_location('match_arquetipos', mod_path)
    match_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(match_mod)
    load_arquetipos = match_mod.load_arquetipos
    match_codes = match_mod.match_codes

# Ajusta la ruta si es necesario
df_arq, df_arq_expl = load_arquetipos('data/arquetipos.csv')

# `df` debe existir en el notebook (resultado de la consulta SQL)
result, merged = match_codes(df, df_arq_expl, sql_code_col='codigo_proceso')
print(f"Filas SQL: {len(df)}; coincidencias en merged (filas con proc_code): {merged['proc_code'].notna().sum()}")
# Filtrar solo filas con coincidencias en arquetipos
result_filtered = result[result['matched_codigo_arquetipo'].notna()]
result_filtered

Filas SQL: 0; coincidencias en merged (filas con proc_code): 0


,id_epica,periodo,codigo_proceso,procedimiento,actividad,matched_proc_codes,matched_codigo_arquetipo,matched_nombre_arquetipo


In [17]:
# Guardar resultados en CSV
import os
from datetime import datetime

# Check if result_filtered exists and has data
if 'result_filtered' not in locals() or result_filtered is None or len(result_filtered) == 0:
	print("Advertencia: No hay resultados con coincidencias para guardar.")
else:
	output_dir = 'data/output'
	os.makedirs(output_dir, exist_ok=True)
	# Usar timestamp para evitar conflictos
	timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
	output_path = os.path.join(output_dir, f'resultado_matching_arquetipos_{timestamp}.csv')
	result_filtered.to_csv(output_path, index=False, encoding='utf-8')
	print(f"Archivo guardado en: {output_path}")
	print(f"Total de filas: {len(result_filtered)}")

Advertencia: No hay resultados con coincidencias para guardar.
